In [1]:
# Check if we have a GPU available
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

GPU available: True
Device: Tesla T4


In [3]:
# Clone your GitHub repo into Colab
# This gives us access to all our scripts
!git clone https://github.com/stargazingpnw/iSpyPlants.git

Cloning into 'iSpyPlants'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 27 (delta 4), reused 27 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 7.86 KiB | 7.86 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [4]:
# Download the Oxford 102 Flowers dataset directly into Colab
# We use wget to download from the official Oxford website
!wget -q https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz
!wget -q https://www.robots.ox.ac.uk/~vgg/data/flowers/102/imagelabels.mat

# Extract the images
!tar -xzf 102flowers.tgz

# Move everything into the right place
!mkdir -p iSpyPlants/data
!mv jpg iSpyPlants/data/
!mv imagelabels.mat iSpyPlants/data/

print("Dataset ready!")

Dataset ready!


In [5]:
# Navigate into the project folder and run our prepare_data script
import os
os.chdir('/content/iSpyPlants')

# Install required packages
!pip install scikit-learn -q

# Run our existing prepare_data script
!python scripts/prepare_data.py

Traceback (most recent call last):
  File "/content/iSpyPlants/scripts/prepare_data.py", line 1, in <module>
    python# prepare_data.py
    ^^^^^^
NameError: name 'python' is not defined


In [6]:
import os
import shutil
from scipy.io import loadmat
from sklearn.model_selection import train_test_split

data_dir = 'data/jpg'
labels_file = 'data/imagelabels.mat'
output_dir = 'data'

labels = loadmat(labels_file)['labels'][0]
images = sorted(os.listdir(data_dir))

train_imgs, valid_imgs, train_labels, valid_labels = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

def copy_images(img_list, label_list, split_name):
    print(f"Organizing {split_name} images...")
    for img, label in zip(img_list, label_list):
        class_dir = os.path.join(output_dir, split_name, str(label))
        os.makedirs(class_dir, exist_ok=True)
        src = os.path.join(data_dir, img)
        dst = os.path.join(class_dir, img)
        shutil.copy(src, dst)
    print(f"Done! {len(img_list)} images organized.")

copy_images(train_imgs, train_labels, 'train')
copy_images(valid_imgs, valid_labels, 'valid')
print("All done! Dataset is ready for training.")

Organizing train images...
Done! 6551 images organized.
Organizing valid images...
Done! 1638 images organized.
All done! Dataset is ready for training.


In [7]:
import os
import torch
from torchvision import datasets, transforms, models
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

valid_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Loading datasets...")
train_dataset = datasets.ImageFolder('data/train', transform=train_transforms)
valid_dataset = datasets.ImageFolder('data/valid', transform=valid_transforms)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=32, shuffle=False)

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(valid_dataset)}")

print("Loading pretrained model...")
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, 102)
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

epochs = 10
best_accuracy = 0

print("Starting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.3f} | Val Accuracy: {accuracy:.2f}%")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        os.makedirs('model', exist_ok=True)
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_to_idx': train_dataset.class_to_idx,
            'best_accuracy': best_accuracy
        }, 'model/best_model.pth')
        print(f"  ✅ New best model saved! ({accuracy:.2f}%)")

print(f"\nTraining complete! Best accuracy: {best_accuracy:.2f}%")

Training on: cuda
Loading datasets...
Training images: 6551
Validation images: 1638
Loading pretrained model...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 193MB/s]


Starting training...
Epoch 1/10 | Loss: 2.614 | Val Accuracy: 81.81%
  ✅ New best model saved! (81.81%)
Epoch 2/10 | Loss: 0.922 | Val Accuracy: 88.10%
  ✅ New best model saved! (88.10%)
Epoch 3/10 | Loss: 0.640 | Val Accuracy: 91.64%
  ✅ New best model saved! (91.64%)
Epoch 4/10 | Loss: 0.513 | Val Accuracy: 92.37%
  ✅ New best model saved! (92.37%)
Epoch 5/10 | Loss: 0.442 | Val Accuracy: 92.86%
  ✅ New best model saved! (92.86%)
Epoch 6/10 | Loss: 0.373 | Val Accuracy: 93.53%
  ✅ New best model saved! (93.53%)
Epoch 7/10 | Loss: 0.350 | Val Accuracy: 93.96%
  ✅ New best model saved! (93.96%)
Epoch 8/10 | Loss: 0.328 | Val Accuracy: 93.65%
Epoch 9/10 | Loss: 0.307 | Val Accuracy: 95.12%
  ✅ New best model saved! (95.12%)
Epoch 10/10 | Loss: 0.292 | Val Accuracy: 94.14%

Training complete! Best accuracy: 95.12%


In [9]:
# Remove model folder from gitignore and allow the .pth file
!echo "!model/best_model.pth" >> .gitignore

# Now add the model file forcefully
!git add -f model/best_model.pth
!git add .gitignore
!git commit -m "Add trained model - 95.12% accuracy"

[main ae0743c] Add trained model - 95.12% accuracy
 2 files changed, 1 insertion(+)
 create mode 100644 model/best_model.pth


In [11]:
# Download the official Oxford flower names file
import json
import scipy.io as sio

# Oxford provides a file mapping class numbers to flower names
!wget -q https://www.robots.ox.ac.uk/~vgg/data/flowers/102/cat_to_name.json 2>/dev/null || echo "not found"

# Let's create our own mapping instead
cat_to_name = {
    "1": "pink primrose", "2": "hard-leaved pocket orchid", "3": "canterbury bells",
    "4": "sweet pea", "5": "english marigold", "6": "tiger lily", "7": "moon orchid",
    "8": "bird of paradise", "9": "monkshood", "10": "globe thistle", "11": "snapdragon",
    "12": "colt's foot", "13": "king protea", "14": "spear thistle", "15": "yellow iris",
    "16": "globe-flower", "17": "purple coneflower", "18": "peruvian lily", "19": "balloon flower",
    "20": "giant white arum lily", "21": "fire lily", "22": "pincushion flower", "23": "fritillary",
    "24": "red ginger", "25": "grape hyacinth", "26": "corn poppy", "27": "prince of wales feathers",
    "28": "stemless gentian", "29": "artichoke", "30": "sweet william", "31": "carnation",
    "32": "garden phlox", "33": "love in the mist", "34": "mexican aster", "35": "alpine sea holly",
    "36": "ruby-lipped cattleya", "37": "cape flower", "38": "great masterwort", "39": "siam tulip",
    "40": "lenten rose", "41": "barbeton daisy", "42": "daffodil", "43": "sword lily",
    "44": "poinsettia", "45": "bolero deep blue", "46": "wallflower", "47": "marigold",
    "48": "buttercup", "49": "oxeye daisy", "50": "common dandelion", "51": "petunia",
    "52": "wild pansy", "53": "primula", "54": "sunflower", "55": "pelargonium",
    "56": "bishop of llandaff", "57": "gaura", "58": "geranium", "59": "orange dahlia",
    "60": "pink-yellow dahlia", "61": "cautleya spicata", "62": "japanese anemone",
    "63": "black-eyed susan", "64": "silverbush", "65": "californian poppy", "66": "osteospermum",
    "67": "spring crocus", "68": "bearded iris", "69": "windflower", "70": "tree poppy",
    "71": "gazania", "72": "azalea", "73": "water lily", "74": "rose", "75": "thorn apple",
    "76": "morning glory", "77": "passion flower", "78": "lotus", "79": "toad lily",
    "80": "anthurium", "81": "frangipani", "82": "clematis", "83": "hibiscus", "84": "columbine",
    "85": "desert-rose", "86": "tree mallow", "87": "magnolia", "88": "cyclamen",
    "89": "watercress", "90": "canna lily", "91": "hippeastrum", "92": "bee balm",
    "93": "ball moss", "94": "foxglove", "95": "bougainvillea", "96": "camellia",
    "97": "mallow", "98": "mexican petunia", "99": "bromelia", "100": "blanket flower",
    "101": "trumpet creeper", "102": "blackberry lily"
}

with open('cat_to_name.json', 'w') as f:
    json.dump(cat_to_name, f)

print("Flower names file created!")
print(f"Class 54 is: {cat_to_name['54']}")

not found
Flower names file created!
Class 54 is: sunflower
